In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ruchi798/bookcrossing-dataset")

print("Path to dataset files:", path)
# /home/kreal/.cache/kagglehub/datasets/ruchi798/bookcrossing-dataset/versions/3


/home/kreal/FastAPI_shop_backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 76.1M/76.1M [01:35<00:00, 834kB/s] 

Extracting files...


Path to dataset files: /home/kreal/.cache/kagglehub/datasets/ruchi798/bookcrossing-dataset/versions/3


In [1]:
import pandas as pd

books = pd.read_csv(
    "/home/kreal/.cache/kagglehub/datasets/ruchi798/bookcrossing-dataset/versions/3/Books Data with Category Language and Summary/Preprocessed_data.csv"
)

In [2]:
import ast
def extract_category(v: list[str]) -> str | None:
    if pd.isna(v):
        return None
    if isinstance(v, list):
        return v[0] if len(v) > 0 else None

    # string that looks like a list
    if isinstance(v, str) and v.startswith("["):
        try:
            parsed = ast.literal_eval(v)
            return parsed[0] if isinstance(parsed, list) and len(parsed) > 0 else None
        except (ValueError, SyntaxError):
            return v

    return v

In [3]:
cols = ['age', 'isbn', 'rating', 'book_title', 'book_author', 'year_of_publication', 'publisher', 'img_s', 'Summary', 'Language', 'Category', 'country']
books_fixed = (
    books[cols]
    .copy()
    .rename(columns={"Summary": "summary", "book_title": "name", "Language": "language", "Category": "category", "img_s": "image"})
    .dropna(subset=["isbn", "book_author", "name", "publisher", "summary", "category"])
)

books_fixed = books_fixed.assign(
    isbn=books_fixed["isbn"].astype("string").str.strip(),
    summary=books_fixed["summary"].astype("string").str.strip(),
    name=books_fixed["name"].astype("string").str.strip(),
    language=books_fixed["language"].astype("string").str.strip(),
    image=books_fixed["image"].astype("string").str.strip(),
    category=books_fixed["category"].apply(extract_category).astype("string").str.strip(),
    age=pd.to_numeric(books_fixed["age"], errors="coerce").round().astype("Int8"),
    rating=pd.to_numeric(books_fixed["rating"], errors="coerce").astype("Float32"),
    year_of_publication=pd.to_numeric(
        books_fixed["year_of_publication"], errors="coerce"
    ).astype("Int16"),
)

books_fixed = books_fixed.drop_duplicates(subset=["isbn"])


In [4]:
books_fixed = books_fixed[~books_fixed["category"].isin(["9"])]

In [5]:
categories_to_be_deleted = books_fixed["category"] \
.value_counts().loc[lambda s: s <= 100] \
.index.to_numpy()
categories_to_be_deleted

array(['Fantasy fiction', 'United States', 'Christmas stories', ...,
       'Microsoft Windows NT.', 'Merchants', 'Alternative histories'],
      shape=(6376,), dtype=object)

In [6]:
books_fixed = books_fixed[~books_fixed["category"].isin(categories_to_be_deleted)]

In [7]:
len(books_fixed)

117629

In [57]:
books_condensed = (
    books_fixed
    .groupby("category", group_keys=False)
    .sample(frac=0.3, random_state=42)
    .reset_index(drop=True)
)

In [58]:
books_condensed = pd.concat(
    [
        books_condensed[books_condensed["category"] != "Fiction"],
        books_condensed[books_condensed["category"] == "Fiction"].sample(4500, random_state=42)
    ],
    ignore_index=True
)

In [59]:
len(books_condensed)

23830

In [60]:
books_condensed.category.value_counts()

category
Fiction                      4500
Juvenile Fiction             3086
Biography & Autobiography    1614
Religion                     1058
History                      1053
                             ... 
African Americans              37
Australia                      37
Design                         35
Adventure and adventurers      34
Great Britain                  32
Name: count, Length: 71, dtype: Int64

In [62]:
books_prices = pd.read_csv("prices.csv")
books_prices["isbn"] = books_prices["isbn"].astype("string").str.strip()
books_prices["price"] = books_prices["price"].astype("Float32")

In [63]:
books_prices.info()

<class 'pandas.DataFrame'>
RangeIndex: 78300 entries, 0 to 78299
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   isbn           78300 non-null  string 
 1   price          78300 non-null  Float32
 2   isbn13         78300 non-null  int64  
 3   original_isbn  78300 non-null  int64  
dtypes: Float32(1), int64(2), string(1)
memory usage: 2.2 MB


In [66]:
books_condensed.drop(columns=["price"], inplace=True, errors="ignore")
books_condensed.head(2)

,age,isbn,rating,name,book_author,year_of_publication,publisher,image,summary,language,category,country
0,43,0884490459,0.0,Preparing for Adolescence: Caution Changes Ahead,Dr. James Dobson,1989,Gospel Light Publications,http://images.amazon.com/images/P/0884490459.0...,Speaks to adolescents about such topics as dru...,en,Adolescence,usa
1,32,0553235311,5.0,Truth about Me & Bobby V (Bantam Sweet Dreams ...,Janetta Johns,1993,Bantam Doubleday Dell Publishing Group,http://images.amazon.com/images/P/0553235311.0...,Candy Davis and her family move from the city ...,en,Adolescence,malaysia


In [67]:
books_prices.head()
books_prices.drop(columns=["isbn13", "original_isbn"], inplace=True, errors="ignore")

In [68]:
books_condensed = books_condensed.merge(books_prices, on="isbn", how="left")

In [73]:
books_condensed["price"] = books_condensed["price"].fillna(books_condensed["price"].mean())

In [74]:
import numpy as np

In [76]:
books_condensed.info()

<class 'pandas.DataFrame'>
RangeIndex: 23830 entries, 0 to 23829
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   age                  23830 non-null  Int8   
 1   isbn                 23830 non-null  string 
 2   rating               23830 non-null  Float32
 3   name                 23830 non-null  string 
 4   book_author          23830 non-null  str    
 5   year_of_publication  23830 non-null  Int16  
 6   publisher            23830 non-null  str    
 7   image                23830 non-null  string 
 8   summary              23830 non-null  string 
 9   language             23830 non-null  string 
 10  category             23830 non-null  string 
 11  country              22756 non-null  str    
 12  price                23830 non-null  Float32
dtypes: Float32(2), Int16(1), Int8(1), str(3), string(6)
memory usage: 2.0 MB


In [82]:
import re
import html

def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = html.unescape(text)           
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [83]:
books_condensed["summary"] = books_condensed["summary"].apply(clean_text)

In [85]:
books_condensed["number_in_stock"] = np.random.randint(30, 150, size=len(books_condensed))
books_condensed["number_in_stock"] = books_condensed["number_in_stock"].astype("Int16")

In [86]:
books_condensed.to_csv("books.csv", index=False)